In [361]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


In [362]:
#import data from /Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/ProcessedObservedData.csv
data = pd.read_csv(r'C:\Users\jaskew\Documents\project_repository\notebooks\observationEventForecasting\DataPreprocessing\FullIndicatorMatrix.csv')
data.drop(columns=['API_UserName','observations','dayofweek','is_weekend','day','month'], inplace=True)
# Ensure the 'date' column is in datetime format
data['date'] = pd.to_datetime(data['date'])
data.head(40)

,date,indicator,seen
0,2025-01-01,185.230.63.171,1
1,2025-01-01,104.21.57.152,0
2,2025-01-01,192.229.221.95,0
3,2025-01-01,216.24.57.253,0
4,2025-01-01,216.24.57.3,0
5,2025-01-01,67.225.140.4,0
6,2025-01-01,104.18.32.68,0
7,2025-01-01,104.26.5.196,0
8,2025-01-01,198.16.66.156,0
9,2025-01-01,198.16.70.28,0


In [363]:
def days_since_last_seen_all(data):
    # Filter only rows where seen == 1
    seen_data = data[data['seen'] == 1]
    
    # Sort the data by indicator and date in descending order
    seen_data = seen_data.sort_values(by=['indicator', 'date'], ascending=[True, False])
    
    # Keep only the most recent entry for each indicator
    seen_data = seen_data.drop_duplicates(subset=['indicator'], keep='first')
    
    # Calculate the difference in days from today for each entry
    today = pd.to_datetime(datetime.now()).normalize()
    seen_data['days_since_last_seen'] = (today - seen_data['date']).dt.days
    
    # Drop rows with NaN values
    seen_data = seen_data.dropna()
    
    return seen_data[['indicator', 'date', 'seen', 'days_since_last_seen']]

# Compute "days since last seen" for all indicators
days_since_last_seen_all_data = days_since_last_seen_all(data)

# Merge the "days since last seen" back into the main dataset
data = data.merge(
    days_since_last_seen_all_data[['indicator', 'days_since_last_seen']],
    on='indicator',
    how='left'
)

# Fill NaN values with a default (e.g., a large number for unseen indicators)
data['days_since_last_seen'] = data['days_since_last_seen'].interpolate(method='linear')
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
data['days_since_last_seen_scaled'] = scaler.fit_transform(data[['days_since_last_seen']])

In [364]:
data.isna().sum()

date                           0
indicator                      0
seen                           0
days_since_last_seen           0
days_since_last_seen_scaled    0
dtype: int64

In [365]:
# Ensure the 'date' column is in datetime format
data['date'] = pd.to_datetime(data['date'])

# Keep the last 7 days in test_data
last_7_days = datetime.now() - timedelta(days=7)
# Normalize today's date (removes the time portion)
today = pd.to_datetime(datetime.now()).normalize()

# Filter test_data: only indicators seen today
test_data = data[(data['date'] == today) & (data['seen'] == 1)]
 
test_data = test_data.reset_index(drop=True)

# Use the remaining data for training
data = data[data['date'] < last_7_days]

# Reset the index of the filtered data
data = data.reset_index(drop=True)
# Display the test_data
test_data

,date,indicator,seen,days_since_last_seen,days_since_last_seen_scaled
0,2025-04-18,185.230.63.171,1,0.0,0.0
1,2025-04-18,185.253.162.21,1,0.0,0.0
2,2025-04-18,64.64.112.131,1,0.0,0.0
3,2025-04-18,64.64.112.146,1,0.0,0.0
4,2025-04-18,156.146.63.165,1,0.0,0.0
5,2025-04-18,162.142.125.242,1,0.0,0.0
6,2025-04-18,162.142.125.255,1,0.0,0.0
7,2025-04-18,162.142.125.247,1,0.0,0.0
8,2025-04-18,146.71.50.198,1,0.0,0.0
9,2025-04-18,104.21.48.1,1,0.0,0.0


In [366]:
# Define the cutoff date (90 days ago from today)
cutoff_date = datetime.now() - timedelta(days=120)

# Convert the date column to datetime format (if not already)
data['date'] = pd.to_datetime(data['date'])

# Filter the data to include only rows within the last 90 days
filtered_data = data[data['date'] >= cutoff_date]
filtered_data

,date,indicator,seen,days_since_last_seen,days_since_last_seen_scaled
0,2025-01-01,185.230.63.171,1,0.0,0.000000
1,2025-01-01,104.21.57.152,0,17.4,0.172277
2,2025-01-01,192.229.221.95,0,34.8,0.344554
3,2025-01-01,216.24.57.253,0,52.2,0.516832
4,2025-01-01,216.24.57.3,0,69.6,0.689109
...,...,...,...,...,...
43021,2025-04-11,47.237.115.193,0,4.0,0.039604
43022,2025-04-11,107.180.119.251,0,3.0,0.029703
43023,2025-04-11,190.92.174.36,0,1.0,0.009901
43024,2025-04-11,192.124.249.112,0,1.0,0.009901


In [367]:
unique_indicators_count = filtered_data['indicator'].nunique()
print(f"Number of unique indicators: {unique_indicators_count}")

Number of unique indicators: 426


In [368]:
target_indicator = '102.129.153.158'
indicator_data = test_data[test_data['indicator'] == target_indicator]
print(indicator_data)


Empty DataFrame
Columns: [date, indicator, seen, days_since_last_seen, days_since_last_seen_scaled]
Index: []


In [369]:
indicator_data = data[data['indicator'] == '102.129.153.158']
seen_counts = indicator_data['seen'].value_counts()
print(seen_counts)



seen
0    99
1     2
Name: count, dtype: int64


In [370]:
# Compute per-indicator rolling 7-day reappearance probability
rolling_prob_list = []

# Process each indicator individually
for indicator, group in filtered_data.groupby('indicator'):
    group = group.sort_values('date').reset_index(drop=True)
    
    # Create target: was it seen 7 days later?
    group['seen_7_days_later'] = group['seen'].shift(-7)
    
    # Rolling mean to estimate probability of reappearance
    group['prob_7_day_reappearance'] = group['seen_7_days_later'].rolling(window=30, min_periods=1).mean()
    
    # Add 'days_since_last_seen' for each row
    group['days_since_last_seen'] = group['date'].apply(
        lambda x: (pd.to_datetime(datetime.now()).normalize() - x).days
    )
    
    # Compute differencing for 'prob_7_day_reappearance'
    group['diff_prob_7_day_reappearance'] = group['prob_7_day_reappearance'].diff()
    
    # Add lagged features
    group['lag_1_prob'] = group['prob_7_day_reappearance'].shift(1)
    group['lag_7_prob'] = group['prob_7_day_reappearance'].shift(7)
    
    # Drop NaN values created by differencing and lagging
    group = group.dropna(subset=['diff_prob_7_day_reappearance', 'lag_1_prob', 'lag_7_prob'])
    
    # Save with indicator name
    group['indicator'] = indicator
    rolling_prob_list.append(group)

# Combine all into one DataFrame
rolling_probs = pd.concat(rolling_prob_list).reset_index(drop=True)

# Ensure 'days_since_last_seen' exists in the main dataset
if 'days_since_last_seen' not in data.columns:
    data['days_since_last_seen'] = data['date'].apply(
        lambda x: (pd.to_datetime(datetime.now()).normalize() - x).days
    )

# Show sample of results
rolling_probs[['date', 'indicator', 'seen', 'prob_7_day_reappearance', 'diff_prob_7_day_reappearance', 'lag_1_prob', 'lag_7_prob', 'days_since_last_seen']].sort_values(
    by='prob_7_day_reappearance', ascending=False
).head(100)

,date,indicator,seen,prob_7_day_reappearance,diff_prob_7_day_reappearance,lag_1_prob,lag_7_prob,days_since_last_seen
1636,2025-02-15,104.21.48.1,1,1.000000,0.000000,1.000000,0.800000,62
1635,2025-02-14,104.21.48.1,1,1.000000,0.033333,0.966667,0.766667,63
1653,2025-03-04,104.21.48.1,1,1.000000,0.000000,1.000000,1.000000,45
1651,2025-03-02,104.21.48.1,1,1.000000,0.000000,1.000000,1.000000,47
1652,2025-03-03,104.21.48.1,1,1.000000,0.000000,1.000000,1.000000,46
...,...,...,...,...,...,...,...,...
1689,2025-04-09,104.21.48.1,1,0.920000,-0.003077,0.923077,0.933333,9
1690,2025-04-10,104.21.48.1,1,0.916667,-0.003333,0.920000,0.933333,8
9216,2025-01-12,162.142.125.255,0,0.916667,0.007576,0.909091,0.800000,96
9228,2025-01-24,162.142.125.255,1,0.916667,-0.039855,0.956522,0.941176,84


In [371]:
rolling_probs.isna().sum()

date                               0
indicator                          0
seen                               0
days_since_last_seen               0
days_since_last_seen_scaled        0
seen_7_days_later               2982
prob_7_day_reappearance            0
diff_prob_7_day_reappearance       0
lag_1_prob                         0
lag_7_prob                         0
dtype: int64

In [372]:
from prophet import Prophet

# Get unique indicators
indicators = rolling_probs['indicator'].unique()

# Store results
forecast_results = []

for ip in indicators:
    ip_data = rolling_probs[rolling_probs['indicator'] == ip].copy()
    
    # Only keep needed columns, drop rows with NaN
    prophet_df = ip_data[['date', 'diff_prob_7_day_reappearance', 'days_since_last_seen', 'lag_1_prob', 'lag_7_prob']].dropna()
    prophet_df = prophet_df.rename(columns={
        'date': 'ds',
        'diff_prob_7_day_reappearance': 'y'  # Use differenced data as the target
    })

    # Skip if not enough history
    if len(prophet_df) < 30:
        continue

    # Cap and floor for logistic growth
    prophet_df['cap'] = 1.0
    prophet_df['floor'] = -1.0  # Adjust floor for differenced data

    try:
        # Define and fit model
        model = Prophet(growth='logistic')
        model.add_regressor('days_since_last_seen')
        model.add_regressor('lag_1_prob')  # Add lagged feature as regressor
        model.add_regressor('lag_7_prob')  # Add lagged feature as regressor
        model.fit(prophet_df)

        # Create future dataframe and extend regressor values
        future = model.make_future_dataframe(periods=7)
        future['cap'] = 1.0
        future['floor'] = -1.0

        # Fill future 'days_since_last_seen' and lagged features with last known values
        future['days_since_last_seen'] = prophet_df['days_since_last_seen'].iloc[-1]
        future['lag_1_prob'] = prophet_df['lag_1_prob'].iloc[-1]
        future['lag_7_prob'] = prophet_df['lag_7_prob'].iloc[-1]

        # Predict
        forecast = model.predict(future)

        # Reverse differencing for predictions
        last_original_value = ip_data['prob_7_day_reappearance'].iloc[-1]
        forecast['yhat'] = forecast['yhat'] + last_original_value

        # Extract 7th day forecast
        day_7_forecast = forecast[['ds', 'yhat']].iloc[-1:].copy()
        day_7_forecast['indicator'] = ip
        forecast_results.append(day_7_forecast)

    except Exception as e:
        print(f"Skipped {ip} due to error: {e}")


# Combine results into one DataFrame
forecast_df = pd.concat(forecast_results).reset_index(drop=True)
forecast_df['yhat_percent'] = (forecast_df['yhat'] * 100).round(2).astype(str) + '%'
forecast_df

10:47:15 - cmdstanpy - INFO - Chain [1] start processing
10:47:16 - cmdstanpy - INFO - Chain [1] done processing
10:47:17 - cmdstanpy - INFO - Chain [1] start processing
10:47:18 - cmdstanpy - INFO - Chain [1] done processing
10:47:18 - cmdstanpy - INFO - Chain [1] start processing
10:47:19 - cmdstanpy - INFO - Chain [1] done processing
10:47:20 - cmdstanpy - INFO - Chain [1] start processing
10:47:21 - cmdstanpy - INFO - Chain [1] done processing
10:47:22 - cmdstanpy - INFO - Chain [1] start processing
10:47:33 - cmdstanpy - INFO - Chain [1] done processing
10:47:34 - cmdstanpy - INFO - Chain [1] start processing
10:47:35 - cmdstanpy - INFO - Chain [1] done processing
10:47:36 - cmdstanpy - INFO - Chain [1] start processing
10:47:47 - cmdstanpy - INFO - Chain [1] done processing
10:47:47 - cmdstanpy - INFO - Chain [1] start processing
10:47:48 - cmdstanpy - INFO - Chain [1] done processing
10:47:49 - cmdstanpy - INFO - Chain [1] start processing
10:48:00 - cmdstanpy - INFO - Chain [1]

KeyboardInterrupt: 

In [ ]:
# Extract the indicators from both dataframes
sorted_results_indicators = set(forecast_df['indicator'])
test_data_indicators = set(test_data['indicator'])

# Find matching indicators
matching_indicators = sorted_results_indicators.intersection(test_data_indicators)

# Find missing indicators in test_data
missing_in_test_data = sorted_results_indicators.difference(test_data_indicators)

# Find missing indicators in sorted_results_df
missing_in_sorted_results = test_data_indicators.difference(sorted_results_indicators)

# Display the results
print("Matching Indicators:", matching_indicators)
print("Indicators in sorted_results_df but missing in test_data:", missing_in_test_data)
print("Indicators in test_data but missing in sorted_results_df:", missing_in_sorted_results)

Matching Indicators: {'162.142.125.247', '162.142.125.255', '68.67.179.164', '104.21.48.1', '64.64.112.131', '146.71.50.198', '185.253.162.21', '185.230.63.171', '64.64.112.146', '156.146.63.165', '162.142.125.242'}
Indicators in sorted_results_df but missing in test_data: {'89.149.38.30', '89.149.38.11', '104.18.32.191', '104.197.253.23', '46.246.8.46', '104.21.2.24', '104.21.90.139', '43.153.103.69', '64.64.112.156', '64.64.112.158', '156.146.63.169', '45.86.200.81', '192.124.249.112', '172.105.94.93', '89.149.38.191', '89.149.38.182', '23.26.221.10', '23.26.221.17', '63.143.42.242', '66.235.168.222', '104.26.5.196', '104.160.6.2', '198.16.70.28', '156.146.63.166', '62.182.98.170', '162.241.225.237', '23.26.221.14', '88.119.174.148', '3.14.182.203', '206.72.195.89', '45.140.146.101', '146.70.204.170', '46.246.8.122', '46.246.8.116', '46.246.8.119', 'hcmiu.edu.vn/', 'pub.marq.com/', '149.88.27.199', '89.149.24.252', '89.149.38.197', '34.160.111.145', '165.231.34.106', '46.246.8.91', '

In [ ]:
precision = len(matching_indicators) / len(sorted_results_indicators)
recall = len(matching_indicators) / len(test_data_indicators)
f1 = 2 * (precision * recall) / (precision + recall)

print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")


Precision: 0.03
Recall: 1.00
F1 Score: 0.05


In [ ]:
# Filter the missing indicators from forecast_df
missing_indicators_data = forecast_df[forecast_df['indicator'].isin(missing_in_test_data)]

# Display the results
missing_indicators_data[['indicator', 'yhat_percent']]

,indicator,yhat_percent
0,102.129.153.158,4.58%
1,102.129.153.43,0.01%
2,102.129.153.71,0.29%
3,102.165.16.161,13.07%
4,103.28.12.56,0.33%
...,...,...
421,pub.marq.com/,22.49%
422,realinvestmentadvice.com/,13.01%
423,www.emergencylighting.com/,-0.32%
424,www.meiersupply.com/,-0.13%


In [ ]:
# Filter the matching indicators from forecast_df
matching_indicators_data = forecast_df[forecast_df['indicator'].isin(matching_indicators)]

# Get the list of indicators from matching_indicators_data
matching_indicators = matching_indicators_data['indicator'].unique()

# Filter the last 90 days of data for these indicators where seen = 1
seen_in_last_90_days = filtered_data[(filtered_data['indicator'].isin(matching_indicators)) & (filtered_data['seen'] == 1)]

# Get the list of indicators that have been seen
seen_indicators = seen_in_last_90_days['indicator'].unique()

# Find indicators that have not been seen in the last 90 days
not_seen_indicators = set(matching_indicators) - set(seen_indicators)

# Display the indicators not seen in the last 90 days
if not_seen_indicators:
    print("The following indicators have NOT been seen in the last 90 days:")
    print(not_seen_indicators)
else:
    print("All matching indicators have been seen in the last 90 days.")

# Exclude not_seen_indicators from the display list
display_indicators = set(matching_indicators) - not_seen_indicators

# Display the results
matching_indicators_data[matching_indicators_data['indicator'].isin(display_indicators)][['indicator', 'yhat_percent']]


All matching indicators have been seen in the last 90 days.


,indicator,yhat_percent
17,104.21.48.1,91.26%
45,146.71.50.198,26.07%
75,156.146.63.165,26.08%
96,162.142.125.242,82.8%
97,162.142.125.247,77.62%
98,162.142.125.255,73.86%
123,185.230.63.171,78.39%
124,185.253.162.21,42.74%
285,64.64.112.131,8.08%
287,64.64.112.146,30.68%


In [ ]:
# Filter records with yhat_percent above 70% and exclude not_seen_indicators
filtered_results = forecast_df[
    (forecast_df['yhat_percent'].str.rstrip('%').astype(float) > 70) & 
    (~forecast_df['indicator'].isin(not_seen_indicators))
].copy()  # Use .copy() to avoid SettingWithCopyWarning

# Add a column to indicate whether the prediction was "Right" or "Wrong"
# Assuming you have a ground truth column 'actual' to compare with 'yhat'
# Replace 'actual' with the appropriate column name in your dataset
filtered_results.loc[:, 'prediction_status'] = filtered_results.apply(
    lambda row: 'Right' if row['yhat'] >= row.get('actual', 0) else 'Wrong',
    axis=1
)

# Identify false negatives: indicators in matching_indicators_data but below the threshold
false_negatives = forecast_df[
    (forecast_df['indicator'].isin(matching_indicators_data['indicator'])) & 
    (forecast_df['yhat_percent'].str.rstrip('%').astype(float) <= 70) &
    (~forecast_df['indicator'].isin(not_seen_indicators))
].copy()  
false_negatives.loc[:, 'prediction_status'] = 'False Negative'

# Identify false positives: indicators not in matching_indicators_data but meet the threshold
false_positives = forecast_df[
    (~forecast_df['indicator'].isin(matching_indicators_data['indicator'])) & 
    (forecast_df['yhat_percent'].str.rstrip('%').astype(float) > 70) &
    (~forecast_df['indicator'].isin(not_seen_indicators))
].copy()  # Use .copy() to avoid SettingWithCopyWarning
false_positives.loc[:, 'prediction_status'] = 'False Positive'

# Combine filtered results, false negatives, and false positives
final_results = pd.concat([filtered_results, false_negatives, false_positives])

# Display the final results
final_results[['indicator', 'yhat_percent', 'prediction_status']]

,indicator,yhat_percent,prediction_status
17,104.21.48.1,91.26%,Right
18,104.21.54.132,78.23%,Right
96,162.142.125.242,82.8%,Right
97,162.142.125.247,77.62%,Right
98,162.142.125.255,73.86%,Right
108,172.240.108.68,82.17%,Right
123,185.230.63.171,78.39%,Right
217,34.160.111.145,86.98%,Right
298,68.67.179.164,76.08%,Right
45,146.71.50.198,26.07%,False Negative


In [ ]:
# Calculate accuracy of final results
# Accuracy = (Number of correct predictions) / (Total number of predictions)
correct_predictions = final_results[final_results['prediction_status'] == 'Right'].shape[0]
total_predictions = final_results.shape[0]
accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0

# Display the accuracy
print (f'{correct_predictions} / {total_predictions}')
print(f"Accuracy: {accuracy:.2%}")

9 / 17
Accuracy: 52.94%


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")  # Suppress warnings for cleaner output

# Define the parameter ranges for grid search
p = d = q = range(0, 2)  # Non-seasonal parameters
P = D = Q = range(0, 2)  # Seasonal parameters
s = [7]  # Seasonal period (weekly seasonality)

# Generate all combinations of parameters
param_combinations = [(x[0], x[1], x[2], x[3], x[4], x[5], x[6]) 
                      for x in [(p_, d_, q_, P_, D_, Q_, s_) 
                                for p_ in p for d_ in d for q_ in q 
                                for P_ in P for D_ in D for Q_ in Q for s_ in s]]

# Function to perform grid search for the best SARIMA parameters
def grid_search_sarima(data, indicator):
    best_score = float("inf")
    best_params = None
    
    # Filter data for the specific indicator
    indicator_data = data[data['indicator'] == indicator]
    indicator_data = indicator_data.sort_values(by='date')
    daily_data = indicator_data.groupby('date')['prob_7_day_reappearance'].mean()
    
    for params in param_combinations:
        try:
            # Fit SARIMA model
            model = SARIMAX(daily_data, 
                            order=(params[0], params[1], params[2]), 
                            seasonal_order=(params[3], params[4], params[5], params[6]))
            result = model.fit(disp=False)
            
            # Calculate mean squared error on the training data
            predictions = result.predict(start=0, end=len(daily_data)-1)
            mse = mean_squared_error(daily_data, predictions)
            
            # Update best parameters if the current model is better
            if mse < best_score:
                best_score = mse
                best_params = params
        except Exception as e:
            continue
    
    return best_params

# Analysis: Extract 7th-day forecast for all indicators and check if they are in test_data
sarima_analysis_results = []

for target_indicator in rolling_probs['indicator'].unique():
    try:
        # Perform grid search to find the best parameters
        best_params = grid_search_sarima(rolling_probs, target_indicator)
        if best_params is None:
            print(f"Skipped indicator {target_indicator} due to no valid parameters.")
            continue

        # Filter data for the specific indicator
        indicator_data = rolling_probs[rolling_probs['indicator'] == target_indicator]
        indicator_data = indicator_data.sort_values(by='date')

        # Aggregate the data to daily frequency
        daily_data = indicator_data.groupby('date')['prob_7_day_reappearance'].mean()

        # Fit the SARIMA model with the best parameters
        sarima_model = SARIMAX(daily_data, 
                               order=(best_params[0], best_params[1], best_params[2]), 
                               seasonal_order=(best_params[3], best_params[4], best_params[5], best_params[6]))
        sarima_result = sarima_model.fit(disp=False)

        # Forecast the next 7 days
        forecast = sarima_result.get_forecast(steps=7)
        forecast_index = forecast.predicted_mean.index
        forecast_values = forecast.predicted_mean
        forecast_conf_int = forecast.conf_int()

        # Check if the indicator is in test_data
        is_in_test_data = target_indicator in test_data['indicator'].unique()

        # Store the 7th-day forecast result
        seventh_day_result = {
            'Indicator': target_indicator,
            'Date': forecast_index[-1],
            'Forecasted Probability': forecast_values.iloc[-1],
            'Lower Bound': forecast_conf_int.iloc[-1, 0],
            'Upper Bound': forecast_conf_int.iloc[-1, 1],
            'In Test Data': 'Y' if is_in_test_data else 'N',
            'Best Params': best_params
        }
        sarima_analysis_results.append(seventh_day_result)       

    except Exception as e:
        print(f"Skipped indicator {target_indicator} due to error: {e}")

# Combine all results into a DataFrame
sarima_forecast_df = pd.DataFrame(sarima_analysis_results)

# Display the forecast results
sarima_forecast_df

KeyboardInterrupt: 

In [ ]:
# Filter records where 'In Test Data' is 'Y'
in_test_data_records = sarima_forecast_df[sarima_forecast_df['In Test Data'] == 'Y']

# Display the filtered records
in_test_data_records

,Indicator,Date,Forecasted Probability,Lower Bound,Upper Bound,In Test Data
13,185.253.162.21,2025-04-18,0.443571,0.328979,0.558163,Y
16,185.230.63.171,2025-04-18,0.784048,0.704035,0.864060,Y
23,104.21.48.1,2025-04-18,0.893510,0.831777,0.955243,Y
237,156.146.63.165,2025-04-18,0.281972,0.199972,0.363973,Y
264,146.71.50.198,2025-04-18,0.266412,0.198779,0.334046,Y
314,162.142.125.247,2025-04-18,0.765639,0.678123,0.853155,Y
318,162.142.125.242,2025-04-18,0.834650,0.733813,0.935487,Y
323,162.142.125.255,2025-04-18,0.723581,0.637327,0.809835,Y
333,64.64.112.131,2025-04-18,0.079704,-0.011949,0.171357,Y
388,64.64.112.146,2025-04-18,0.313861,0.233580,0.394143,Y


In [ ]:
filtered_records = sarima_forecast_df[(sarima_forecast_df['Upper Bound'] > .50) & (sarima_forecast_df['In Test Data'] == 'N')]

# Display the filtered records
filtered_records

,Indicator,Date,Forecasted Probability,Lower Bound,Upper Bound,In Test Data
22,104.21.54.132,2025-04-18,0.770271,0.694802,0.845739,N
69,172.240.108.68,2025-04-18,0.803848,0.703668,0.904027,N
91,104.21.61.32,2025-04-18,0.529747,0.428830,0.630665,N
96,178.175.129.37,2025-04-18,0.393190,0.282606,0.503774,N
177,34.160.111.145,2025-04-18,0.873719,0.788559,0.958879,N
195,91.90.122.24,2025-04-18,0.440551,0.336110,0.544992,N
276,www.shorturl.at/,2025-04-18,0.484410,0.396989,0.571831,N
